In [1]:
import os
import sys
project_path = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_path)

requirements_path = os.path.join(project_path, "SECONDARY/requirements.txt")
!{sys.executable} -m pip install -r "{requirements_path}"

  Using cached binance_connector-3.12.0-py3-none-any.whl.metadata (13 kB)
  Using cached binance_historical_data-0.1.14-py3-none-any.whl.metadata (8.3 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Using cached jupyter_dash-0.4.2-py3-none-any.whl.metadata (3.6 kB)
  Using cached ipylab-1.1.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached mplfinance-0.12.10b0-py3-none-any.whl.metadata (19 kB)
  Using cached torch-2.0.1-cp310-none-macosx_10_9_x86_64.whl.metadata (23 kB)
  Using cached torchvision-0.17.2-cp310-cp310-macosx_10_13_x86_64.whl.metadata (6.6 kB)
  Using cached torchaudio-2.2.2-cp310-cp310-macosx_10_13_x86_64.whl.metadata (6.4 kB)
  Using cached tradingview_ta-3.3.0-py3-none-any.whl.metadata (4.6 kB)
  Using cached pyspark-3.5.6.tar.gz (317.4 MB)
  Preparing metadata (setup.py) 

In [3]:
import os
import sys
import time
!{sys.executable} --version
if sys.version_info.minor == 8:
    raise RuntimeError('USE JUPYTER KERNEL VENV 3.10/310/DEFAULT INSTEAD')

!cd /workspace/CRYPTO_MACAQUES && pip install .
!cd /home/crypto/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && . venv/bin/activate && !{sys.executable} -m pip install .
!cd $HOME/Documents/GitHub/CRYPTO_MACAQUES && python3 -m venv venv && source venv/bin/activate && !{sys.executable} -m pip install .

from IPython.core.display_functions import clear_output
clear_output()

In [4]:
%load_ext autoreload
%autoreload

from dateutil import parser
from SRC.LIBRARIES.new_data_utils import fetch

symbol = 'BTCUSDT'
market_type = 'SPOT'
discretization = '1D'
start_dt = parser.parse('2024-01-01')

df = fetch(market_type, symbol, discretization, start_dt)
print(df.head())

VALIDATED: BTCUSDT | utc_timestamp | 1D | count = 1 | unique = [0] | first_utc_dt = 00:00 02-01-24 | last_utc_dt = 00:00 20-03-26
                            symbol discretization            kiev_timestamp  \
timestamp                                                                     
2024-01-02 00:00:00+00:00  BTCUSDT             1D 2024-01-02 02:00:00+02:00   
2024-01-03 00:00:00+00:00  BTCUSDT             1D 2024-01-03 02:00:00+02:00   
2024-01-04 00:00:00+00:00  BTCUSDT             1D 2024-01-04 02:00:00+02:00   
2024-01-05 00:00:00+00:00  BTCUSDT             1D 2024-01-05 02:00:00+02:00   
2024-01-06 00:00:00+00:00  BTCUSDT             1D 2024-01-06 02:00:00+02:00   

                                      utc_timestamp      open      high  \
timestamp                                                                 
2024-01-02 00:00:00+00:00 2024-01-02 00:00:00+00:00  42283.58  44184.10   
2024-01-03 00:00:00+00:00 2024-01-03 00:00:00+00:00  44179.55  45879.63   
2024-01-04 00:00

In [5]:
import ta
import numpy as np
#Process iteration over timeseries dataframe with sliding window approach
window_size = 30

def mrc_supersmoother(src, length):
    """
    Supersmoother filter by John Ehlers
    """
    a1 = np.exp(-1.414 * np.pi / length)
    b1 = 2 * a1 * np.cos(1.414 * np.pi / length)
    c3 = -a1 * a1
    c2 = b1
    c1 = 1 - c2 - c3

    ss = np.zeros_like(src)
    ss[:2] = src[:2]

    for i in range(2, len(src)):
        ss[i] = c1 * src[i] + c2 * ss[i-1] + c3 * ss[i-2]

    return ss

def mrc_sak_filter(filter_type, src, length):
    """
    Ehlers Swiss Army Knife filters
    """
    pi = np.pi
    cycle = 2 * pi / length

    c0, c1 = 1.0, 0.0
    b0, b1, b2 = 1.0, 0.0, 0.0
    a1, a2 = 0.0, 0.0
    alpha, beta, gamma = 0.0, 0.0, 0.0

    if filter_type == "Ehlers EMA":
        alpha = (np.cos(cycle) + np.sin(cycle) - 1) / np.cos(cycle)
        b0 = alpha
        a1 = 1 - alpha

    elif filter_type == "Gaussian":
        beta = 2.415 * (1 - np.cos(cycle))
        alpha = -beta + np.sqrt(beta * beta + 2 * beta)
        c0 = alpha * alpha
        a1 = 2 * (1 - alpha)
        a2 = -(1 - alpha) * (1 - alpha)

    elif filter_type == "Butterworth":
        beta = 2.415 * (1 - np.cos(cycle))
        alpha = -beta + np.sqrt(beta * beta + 2 * beta)
        c0 = alpha * alpha / 4
        b1, b2 = 2, 1
        a1 = 2 * (1 - alpha)
        a2 = -(1 - alpha) * (1 - alpha)

    elif filter_type == "BandStop":
        beta = np.cos(cycle)
        gamma = 1 / np.cos(cycle * 2 * 0.1)  # delta = 0.1
        alpha = gamma - np.sqrt(gamma * gamma - 1)
        c0 = (1 + alpha) / 2
        b1 = -2 * beta
        b2 = 1
        a1 = beta * (1 + alpha)
        a2 = -alpha

    elif filter_type == "SMA":
        c1 = 1 / length
        b0 = 1 / length
        a1 = 1

    elif filter_type == "EMA":
        alpha = 2 / (length + 1)
        b0 = alpha
        a1 = 1 - alpha

    elif filter_type == "RMA":
        alpha = 1 / length
        b0 = alpha
        a1 = 1 - alpha

    output = np.zeros_like(src)
    output[:3] = src[:3]

    for i in range(3, len(src)):
        output[i] = (c0 * (b0 * src[i] +
                          b1 * (src[i-1] if i-1 >= 0 else 0) +
                          b2 * (src[i-2] if i-2 >= 0 else 0)) +
                    a1 * output[i-1] +
                    a2 * output[i-2] -
                    c1 * (src[i-length] if i-length >= 0 else 0))

    return output

def mrc_calculate(df, source_type='hlc3', filter_type='SuperSmoother',
                  length=200, innermult=1.0, outermult=2.415):
    """
    Calculate Mean Reversion Channel

    Parameters:
    df : DataFrame with 'high', 'low', 'close' columns
    source_type : 'hlc3', 'ohlc4', or 'close'
    filter_type : Type of filter to use
    length : Lookback period
    innermult : Inner channel multiplier
    outermult : Outer channel multiplier
    """

    # Calculate source price
    if source_type == 'hlc3':
        source = (df['high'] + df['low'] + df['close']) / 3
    elif source_type == 'ohlc4':
        source = (df['open'] + df['high'] + df['low'] + df['close']) / 4
    else:  # 'close'
        source = df['close']

    source = source.values

    # Calculate True Range
    tr = np.maximum(
        df['high'] - df['low'],
        np.maximum(
            abs(df['high'] - df['close'].shift(1)),
            abs(df['low'] - df['close'].shift(1))
        )
    ).fillna(0).values

    # Apply filters
    if filter_type == 'SuperSmoother':
        meanline = mrc_supersmoother(source, length)
        meanrange = mrc_supersmoother(tr, length)
    else:
        meanline = mrc_sak_filter(filter_type, source, length)
        meanrange = mrc_sak_filter(filter_type, tr, length)

    pi = np.pi
    mult = pi * innermult
    mult2 = pi * outermult

    # Calculate channels
    upband1 = meanline + (meanrange * mult)
    loband1 = meanline - (meanrange * mult)
    upband2 = meanline + (meanrange * mult2)
    loband2 = meanline - (meanrange * mult2)

    # Calculate condition (optional)
    condition = mrc_calculate_condition(df, meanline, meanrange, upband2, loband2)

    return {
        'meanline': meanline,
        'meanrange': meanrange,
        'upband1': upband1,
        'loband1': loband1,
        'upband2': upband2,
        'loband2': loband2,
        'condition': condition
    }

def mrc_calculate_condition(df, meanline, meanrange, upband2, loband2):
    """
    Calculate MRC condition (overbought/oversold status)
    """
    gradsize = 0.5
    condition = np.zeros(len(df))

    high = df['high'].values
    low = df['low'].values
    close = df['close'].values

    for i in range(len(df)):
        if close[i] > meanline[i]:
            upband2_1 = upband2[i] + (meanrange[i] * gradsize * 4)
            upband2_9 = upband2[i] + (meanrange[i] * gradsize * -4)

            if high[i] >= upband2_9 and high[i] < upband2[i]:
                condition[i] = 1  # Overbought (Weak)
            elif high[i] >= upband2[i] and high[i] < upband2_1:
                condition[i] = 2  # Overbought
            elif high[i] >= upband2_1:
                condition[i] = 3  # Overbought (Strong)
            elif close[i] <= meanline[i] + meanrange[i]:
                condition[i] = 4  # Price Near Mean
            else:
                condition[i] = 5  # Price Above Mean

        elif close[i] < meanline[i]:
            loband2_1 = loband2[i] - (meanrange[i] * gradsize * 4)
            loband2_9 = loband2[i] - (meanrange[i] * gradsize * -4)

            if low[i] <= loband2_9 and low[i] > loband2[i]:
                condition[i] = -1  # Oversold (Weak)
            elif low[i] <= loband2[i] and low[i] > loband2_1:
                condition[i] = -2  # Oversold
            elif low[i] <= loband2_1:
                condition[i] = -3  # Oversold (Strong)
            elif close[i] >= meanline[i] + meanrange[i]:
                condition[i] = -4  # Price Near Mean
            else:
                condition[i] = -5  # Price Below Mean

    return condition

def stochastic_tradingview(high, low, close, periodK=14, smoothK=3, periodD=3):
    lowest_low = low.rolling(window=periodK).min()
    highest_high = high.rolling(window=periodK).max()
    raw_k = 100 * (close - lowest_low) / (highest_high - lowest_low)
    stoch_k = raw_k.rolling(window=smoothK).mean()
    stoch_d = stoch_k.rolling(window=periodD).mean()

    return stoch_k, stoch_d

df['rsi'] = ta.momentum.RSIIndicator(close=df['close'], window=14).rsi()

df['stoch_k'], df['stoch_d'] = stochastic_tradingview(
    high=df['high'],
    low=df['low'],
    close=df['close'])

macd = ta.trend.MACD(close=df['close'], window_slow=26, window_fast=12, window_sign=9)
df['macd_line'] = macd.macd()
df['macd_signal'] = macd.macd_signal()
df['macd_histogram'] = macd.macd_diff()

df['atr'] = ta.volatility.AverageTrueRange(
    high=df['high'],
    low=df['low'],
    close=df['close'],
    window=14).average_true_range()

for i in range(len(df) - window_size + 1):
    window = df.iloc[i:i + window_size]
    # process window
    print(f"{window.index[0]} - {window.index[-1]}")

2024-01-02 00:00:00+00:00 - 2024-01-31 00:00:00+00:00
2024-01-03 00:00:00+00:00 - 2024-02-01 00:00:00+00:00
2024-01-04 00:00:00+00:00 - 2024-02-02 00:00:00+00:00
2024-01-05 00:00:00+00:00 - 2024-02-03 00:00:00+00:00
2024-01-06 00:00:00+00:00 - 2024-02-04 00:00:00+00:00
2024-01-07 00:00:00+00:00 - 2024-02-05 00:00:00+00:00
2024-01-08 00:00:00+00:00 - 2024-02-06 00:00:00+00:00
2024-01-09 00:00:00+00:00 - 2024-02-07 00:00:00+00:00
2024-01-10 00:00:00+00:00 - 2024-02-08 00:00:00+00:00
2024-01-11 00:00:00+00:00 - 2024-02-09 00:00:00+00:00
2024-01-12 00:00:00+00:00 - 2024-02-10 00:00:00+00:00
2024-01-13 00:00:00+00:00 - 2024-02-11 00:00:00+00:00
2024-01-14 00:00:00+00:00 - 2024-02-12 00:00:00+00:00
2024-01-15 00:00:00+00:00 - 2024-02-13 00:00:00+00:00
2024-01-16 00:00:00+00:00 - 2024-02-14 00:00:00+00:00
2024-01-17 00:00:00+00:00 - 2024-02-15 00:00:00+00:00
2024-01-18 00:00:00+00:00 - 2024-02-16 00:00:00+00:00
2024-01-19 00:00:00+00:00 - 2024-02-17 00:00:00+00:00
2024-01-20 00:00:00+00:00 - 

In [7]:
%load_ext autoreload
%autoreload

import plotly.graph_objects as go
from plotly.subplots import make_subplots

candlestick_row = 1
volume_row = 2
rsi_row = 3
stoch_row = 4
macd_row = 5
atr_row = 6

def add_bars(col, name, color, row):
    fig.add_trace(
        go.Bar(
            x=df.index,
            y=df[col],
            name=name,
            marker=dict(color=color),
            width=(df.index[1] - df.index[0]).total_seconds() * 1000,
        ),
        row=row, col=1
)

def add_scatter(col, name, color, row, fill=None, fillcolor=None, width=2, dash=None):
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df[col],
            name=name,
            line=dict(color=color, width=width, dash=dash),
            mode='lines',
            fill=fill,
            fillcolor=fillcolor
        ),
        row=row, col=1
    )

def add_over_zone(y0, y1, fillcolor, row):
    fig.add_hrect(
        y0=y0, y1=y1,
        line_width=0,
        fillcolor=fillcolor,
        opacity=0.2,
        row=row, col=1
    )

def add_central_line(row, y=50, line_dash="dot"):
    fig.add_hline(
        y=y,
        line_dash=line_dash,
        line_color="white",
        line_width=1,
        opacity=0.3,
        row=row, col=1
    )

def add_over_zones_and_a_central_line(row):
    add_over_zone(80, 100, "red", row)
    add_over_zone(0, 20, "green", row)
    add_central_line(row)

def add_stoch(speed, color):
    add_scatter('stoch_' + speed, "Stoch %" + speed.capitalize(), color, stoch_row)

fig = make_subplots(
    rows=6, cols=1,
    shared_xaxes=True,
    row_heights=[15, 1, 3, 3, 3, 2],
    vertical_spacing=0.03
)

fig.add_trace(
    go.Candlestick(
        x=df.index,
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="OHLC"
    ),
    row=candlestick_row, col=1
)

mrc_result = mrc_calculate(
    df,
    source_type='hlc3',
    filter_type='SuperSmoother',
    length=200,
    innermult=1.0,
    outermult=2.415
)
df['meanline'] = mrc_result['meanline']
df['upband1'] = mrc_result['upband1']
df['loband1'] = mrc_result['loband1']
df['upband2'] = mrc_result['upband2']
df['loband2'] = mrc_result['loband2']
add_scatter('meanline', "MRC Mean", '#FFCD00', candlestick_row)
add_scatter('upband1', "MRC R1", 'green', candlestick_row, width=1, dash='dot')
add_scatter('loband1', "MRC S1", 'green', candlestick_row, width=1, dash='dot')
add_scatter('upband2', "MRC R2", 'red', candlestick_row, width=1)
add_scatter('loband2', "MRC S2", 'red', candlestick_row, width=1)

add_bars(
    "volume",
    "Volume",
    ["green" if c > o else "red" for o, c in zip(df["open"], df["close"])],
    volume_row)

add_scatter('rsi', "RSI", 'purple', rsi_row)
add_over_zones_and_a_central_line(rsi_row)

add_stoch('k', "lightblue")
add_stoch('d', "orange")
add_over_zones_and_a_central_line(stoch_row)

prev_macd = df['macd_histogram'].shift(1)
macd_colors = [
    'green' if (val >= 0 and (i == 0 or val >= prev_macd.iloc[i]))
    else 'lightgreen' if (val >= 0 and val < prev_macd.iloc[i])
    else 'red' if (val < 0 and (i == 0 or val <= prev_macd.iloc[i]))
    else 'lightcoral' if (val < 0 and val > prev_macd.iloc[i])
    else 'rgba(128, 128, 128, 0.3)'
    for i, val in enumerate(df['macd_histogram'])
]
add_scatter("macd_line", "MACD Line", 'lightblue', macd_row)
add_scatter("macd_signal", "Signal Line", 'orange', macd_row)
add_bars("macd_histogram", "MACD Histogram", macd_colors, macd_row)
add_central_line(macd_row, line_dash="solid")

add_scatter('atr', "ATR (14)", 'orange', atr_row, fill='tozeroy', fillcolor='rgba(255, 165, 0, 0.1)')
add_central_line(atr_row, y=df['atr'].mean())

fig.update_layout(bargap=0)

fig.update_layout(
    title="OHLC with Volume Explosion After Valley",
    xaxis_rangeslider_visible=False,
    yaxis_title="Price",
    yaxis2_title="Volume",
    yaxis3_title="RSI",
    yaxis4_title="Stoch",
    yaxis5_title="MACD",
    yaxis6_title="ATR",
    height=3000
)

fig.show()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
